# NB_06 — GDT × Multiplexed Readout v1.1

**Research question**

> Can the discrete SMuRF / µMUX readout constraints represented in `SOURCE_05` be converted into a physically justified General Divisor Theorem specification of admissible readout states?

This notebook is deliberately stricter than a generic integer-search notebook. It accepts a candidate GDT mapping only if all four theorem inputs have an explicit engineering meaning:

\[
n,\qquad m,\qquad a,\qquad N
\]

with

\[
n \equiv a \pmod m,
\qquad
\gcd(n,N)=1.
\]

The notebook rejects mappings that arise only from arbitrary indexing, unit conversion, or convenient numerical factorization.


## Current discrete evidence from SOURCE_05

The readout architecture contains real discrete structure:

- 528 resonators assembled as 8 × 66-channel chips;
- four resonator sub-bands per 66-channel chip;
- 128 overlapping digital analysis sub-bands;
- up to four tones per analysis sub-band;
- tracked Fourier harmonic indices \(n=1,2,3\);
- channel quality enable/disable states;
- physical collision and band-edge exclusions.

These facts make multiplexed readout a stronger GDT candidate than continuous electroplating variables, but they do not by themselves establish a residue-class plus coprimality rule.


## 1. Locate repository and load SOURCE_05 + Engineering Object


In [ ]:
from __future__ import annotations

from math import gcd, lcm
from pathlib import Path
import json
import shutil
import subprocess
import sys
import zipfile

import pandas as pd
import yaml

REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
REPO_ROOT_OVERRIDE = None

NOTEBOOK_ID = "NB_06_GDT_MULTIPLEXED_READOUT"
AUDIT_ID = "GDT_READOUT_01"

SOURCE_FILENAME = "SOURCE_05_smurf_multiplexed_readout.yaml"
OBJECT_FILENAME = "multiplexed_readout.yaml"


def find_repo_root() -> Path:
    candidates = []

    if REPO_ROOT_OVERRIDE is not None:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser().resolve())

    start = Path.cwd().resolve()
    candidates.extend([start, *start.parents])
    candidates.extend([
        Path("/content/sensors-becker"),
        Path("/home/dan/sensors-becker"),
        Path.home() / "sensors-becker",
    ])

    for candidate in candidates:
        if candidate.is_dir() and (candidate / "engineering_navigator").is_dir():
            return candidate

    if Path("/content").exists():
        target = Path("/content/sensors-becker")
        if not target.exists():
            subprocess.run(
                ["git", "clone", REPOSITORY_URL, str(target)],
                check=True,
            )
        if (target / "engineering_navigator").is_dir():
            return target

    raise FileNotFoundError(
        "Could not locate sensors-becker. Set REPO_ROOT_OVERRIDE explicitly."
    )


ROOT = find_repo_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SOURCE_PATH = (
    ROOT
    / "engineering_navigator"
    / "multiplexed_readout"
    / "source_records"
    / SOURCE_FILENAME
)

OBJECT_PATH = (
    ROOT
    / "engineering_navigator"
    / "engineering_objects"
    / OBJECT_FILENAME
)

OUTPUT_DIR = (
    ROOT
    / "outputs"
    / "engineering_questions"
    / "multiplexed_readout"
    / AUDIT_ID
)

EXPORT_DIR = ROOT / "exports" / AUDIT_ID
EXPORT_ZIP = ROOT / "exports" / f"{AUDIT_ID}_export.zip"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)

print("Repository:", ROOT)
print("SOURCE_05 :", SOURCE_PATH.relative_to(ROOT))
print("Object    :", OBJECT_PATH.relative_to(ROOT))


## 2. Load and validate current readout evidence


In [ ]:
def load_yaml(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(path)

    data = yaml.safe_load(path.read_text(encoding="utf-8"))

    if not isinstance(data, dict):
        raise TypeError(f"{path}: expected one top-level YAML mapping")

    return data


source = load_yaml(SOURCE_PATH)
readout_object = load_yaml(OBJECT_PATH)

if source.get("source_id") != "SOURCE_05":
    raise ValueError("Expected SOURCE_05")

if readout_object.get("id") != "multiplexed_readout":
    raise ValueError("Expected multiplexed_readout Engineering Object")

print("SOURCE status:", source.get("extraction_status"))
print("Object status:", readout_object.get("object_status"))
print("GDT source status:", source.get("gdt_applicability", {}).get("status"))


## 3. General Divisor Theorem computational implementation


In [ ]:
def prime_factors(n: int) -> set[int]:
    if n <= 0:
        raise ValueError("n must be positive")

    factors = set()
    x = n
    p = 2

    while p * p <= x:
        while x % p == 0:
            factors.add(p)
            x //= p
        p += 1

    if x > 1:
        factors.add(x)

    return factors


def radical(n: int) -> int:
    result = 1
    for p in prime_factors(n):
        result *= p
    return result


def phi(n: int) -> int:
    result = n
    for p in prime_factors(n):
        result -= result // p
    return result


def gdt_parameters(N: int, m: int, a: int) -> dict:
    if N <= 0 or m <= 0:
        raise ValueError("N and m must be positive")

    d = radical(gcd(m, N))
    R = radical(N) // d
    admissible_branch = gcd(a, d) == 1

    return {
        "N": N,
        "m": m,
        "a": a,
        "d": d,
        "R": R,
        "admissible_branch": admissible_branch,
        "minimal_period": lcm(m, radical(N)),
        "accepted_per_period": phi(R) if admissible_branch else 0,
        "correction_factor": d / phi(d) if admissible_branch else None,
    }


def gdt_states(N: int, m: int, a: int, L: int) -> list[int]:
    params = gdt_parameters(N, m, a)

    if not params["admissible_branch"]:
        return []

    return [
        n
        for n in range(1, L + 1)
        if n % m == a % m and gcd(n, N) == 1
    ]


# Mathematical control only — not a TES readout result.
print(gdt_parameters(30, 6, 1))
print(gdt_states(30, 6, 1, 30))


## 4. Extract discrete readout structures


In [ ]:
candidate_structures = (
    source.get("gdt_applicability", {})
    .get("candidate_integer_structures", [])
)

structure_rows = []

for item in candidate_structures:
    structure_rows.append(
        {
            "structure": item.get("structure"),
            "evidence": item.get("evidence"),
            "gdt_status": item.get("gdt_status"),
            "source_page": item.get("source_page"),
            "source_pages": item.get("source_pages"),
        }
    )

discrete_structures = pd.DataFrame(structure_rows)
discrete_structures


## 5. Candidate GDT mappings

Each candidate receives one of four states:

- `physically_justified`
- `interesting_but_incomplete`
- `arbitrary_encoding`
- `incompatible_with_source`

Only `physically_justified` candidates are eligible for theorem evaluation.


In [ ]:
candidate_mappings = [
    {
        "candidate_id": "READOUT_GDT_001",
        "name": "analysis_subband_index",
        "state_variable": "analysis filter-bank sub-band index",
        "integer_domain_supported": True,
        "candidate_m": 128,
        "candidate_a": None,
        "candidate_N": None,
        "integer_basis": "128 discrete analysis sub-bands are explicitly reported.",
        "residue_basis": "",
        "coprimality_basis": "",
        "status": "interesting_but_incomplete",
        "reason": (
            "Integer indexing is real, but SOURCE_05 provides no physical rule selecting "
            "one residue class modulo 128 and no factor-exclusion/coprimality rule."
        ),
    },
    {
        "candidate_id": "READOUT_GDT_002",
        "name": "tones_per_subband",
        "state_variable": "tone count within one analysis sub-band",
        "integer_domain_supported": True,
        "candidate_m": 4,
        "candidate_a": None,
        "candidate_N": None,
        "integer_basis": "Firmware supports up to four tones per analysis sub-band.",
        "residue_basis": "",
        "coprimality_basis": "",
        "status": "interesting_but_incomplete",
        "reason": "A bounded capacity of four is not itself a residue-class rule.",
    },
    {
        "candidate_id": "READOUT_GDT_003",
        "name": "tracked_harmonic_order",
        "state_variable": "Fourier harmonic order",
        "integer_domain_supported": True,
        "candidate_m": 3,
        "candidate_a": None,
        "candidate_N": None,
        "integer_basis": "The first three harmonics n=1,2,3 are explicitly tracked.",
        "residue_basis": "",
        "coprimality_basis": "",
        "status": "interesting_but_incomplete",
        "reason": (
            "The integer harmonic index is physically meaningful, but the source does not "
            "specify an admissible residue class or coprimality condition."
        ),
    },
    {
        "candidate_id": "READOUT_GDT_004",
        "name": "resonator_grouping",
        "state_variable": "resonator position/group index within a 66-channel chip",
        "integer_domain_supported": True,
        "candidate_m": 4,
        "candidate_a": None,
        "candidate_N": None,
        "integer_basis": (
            "Each 66-channel chip is intentionally grouped into four resonator sub-bands "
            "to reduce intra-chip collision risk."
        ),
        "residue_basis": "",
        "coprimality_basis": "",
        "status": "interesting_but_incomplete",
        "reason": (
            "The grouping is physically motivated, but SOURCE_05 does not report a modular "
            "assignment formula or factor-exclusion law on resonator indices."
        ),
    },
    {
        "candidate_id": "READOUT_GDT_005",
        "name": "frequency_grid_by_khz_scaling",
        "state_variable": "frequency converted to integer kHz bins",
        "integer_domain_supported": False,
        "candidate_m": 500000,
        "candidate_a": None,
        "candidate_N": None,
        "integer_basis": (
            "This would arise only by multiplying GHz/MHz frequencies into an integer grid."
        ),
        "residue_basis": "",
        "coprimality_basis": "",
        "status": "arbitrary_encoding",
        "reason": (
            "Unit conversion creates integers but does not create a physical GDT state."
        ),
    },
    {
        "candidate_id": "READOUT_GDT_006",
        "name": "flux_quanta_per_ramp",
        "state_variable": "flux quanta swept per ramp period",
        "integer_domain_supported": False,
        "candidate_m": None,
        "candidate_a": None,
        "candidate_N": None,
        "integer_basis": "",
        "residue_basis": "",
        "coprimality_basis": "",
        "status": "incompatible_with_source",
        "reason": (
            "SOURCE_05 reports approximate values such as 3.3 and 3.4 Phi0, "
            "so the quantity should not be promoted to an integer state."
        ),
    },
]

candidate_mapping_df = pd.DataFrame(candidate_mappings)

candidate_mapping_df[
    ["candidate_id", "name", "status", "reason"]
]


## 6. Direct-hypothesis checklist


In [ ]:
hypothesis_rows = []

for candidate in candidate_mappings:
    hypothesis_rows.append(
        {
            "candidate_id": candidate["candidate_id"],
            "integer_state_n": bool(candidate.get("integer_domain_supported")),
            "natural_modulus_m": bool(candidate.get("candidate_m")),
            "physical_residue_rule": bool(candidate.get("residue_basis")),
            "physical_coprimality_rule": bool(candidate.get("coprimality_basis")),
            "direct_gdt_ready": (
                candidate.get("status") == "physically_justified"
                and bool(candidate.get("integer_domain_supported"))
                and bool(candidate.get("candidate_m"))
                and bool(candidate.get("residue_basis"))
                and bool(candidate.get("coprimality_basis"))
            ),
        }
    )

hypothesis_audit = pd.DataFrame(hypothesis_rows)
hypothesis_audit


## 7. Evaluate only physically justified mappings


In [ ]:
gdt_results = []

for candidate in candidate_mappings:
    if candidate["status"] != "physically_justified":
        continue

    N = int(candidate["candidate_N"])
    m = int(candidate["candidate_m"])
    a = int(candidate["candidate_a"])
    L = int(candidate.get("evaluation_limit_L", 0))

    params = gdt_parameters(N, m, a)
    states = gdt_states(N, m, a, L)

    gdt_results.append(
        {
            "candidate_id": candidate["candidate_id"],
            "name": candidate["name"],
            **params,
            "evaluation_limit_L": L,
            "accepted_state_count": len(states),
            "accepted_states": states,
        }
    )

gdt_result_columns = [
    "candidate_id",
    "name",
    "N",
    "m",
    "a",
    "d",
    "R",
    "admissible_branch",
    "minimal_period",
    "accepted_per_period",
    "correction_factor",
    "evaluation_limit_L",
    "accepted_state_count",
    "accepted_states",
]

gdt_result_df = pd.DataFrame(
    gdt_results,
    columns=gdt_result_columns,
)

if gdt_result_df.empty:
    print("No physically justified direct GDT mapping is currently available.")
else:
    gdt_result_df


## 8. Physical exclusions versus arithmetic exclusions

SOURCE_05 already provides strong physical exclusion rules:

- resonators near or closer than approximately one linewidth can collide;
- resonators near 500 MHz band edges may be untrackable;
- channels with abnormal tracking/IQ response are disabled.

The next question is whether any of these exclusions can be represented exactly, not approximately, as arithmetic factor exclusion.


In [ ]:
constraint_rows = []

for item in source.get("engineering_constraints", []):
    constraint_rows.append(
        {
            "constraint": item.get("constraint"),
            "statement": item.get("statement"),
            "source_page": item.get("source_page"),
            "candidate_for_exact_factor_exclusion": (
                item.get("constraint")
                in {
                    "resonator_collision",
                    "band_edge_exclusion",
                    "channel_quality_cut",
                }
            ),
            "currently_expressible_as_gcd_rule": False,
        }
    )

physical_exclusion_audit = pd.DataFrame(constraint_rows)
physical_exclusion_audit


## 9. What evidence would convert the best candidate into a direct GDT application?


In [ ]:
next_evidence = pd.DataFrame([
    {
        "candidate": "analysis_subband_index",
        "missing_rule": "physical residue-class assignment",
        "question": (
            "Does firmware assign allowed channel/sub-band indices by a repeating integer rule "
            "rather than by arbitrary table lookup?"
        ),
        "needed_evidence": (
            "Firmware or design documentation giving an explicit periodic index-allocation rule."
        ),
    },
    {
        "candidate": "analysis_subband_index",
        "missing_rule": "physical factor-exclusion rule",
        "question": (
            "Are specific sub-band/channel indices excluded because they share factors with "
            "a hardware/synchronization integer N?"
        ),
        "needed_evidence": (
            "A hardware, clocking, aliasing, or synchronization rule exactly equivalent to gcd(n,N)=1."
        ),
    },
    {
        "candidate": "resonator_grouping",
        "missing_rule": "explicit modular placement formula",
        "question": (
            "Are resonator group assignments generated from channel index modulo a fixed group count?"
        ),
        "needed_evidence": (
            "Mask-layout or frequency-plan documentation specifying the assignment formula."
        ),
    },
    {
        "candidate": "resonator_grouping",
        "missing_rule": "exact collision exclusion",
        "question": (
            "Can collision avoidance be reduced from a frequency-distance inequality to an exact "
            "integer factor-exclusion condition?"
        ),
        "needed_evidence": (
            "An actual design rule showing factor-based exclusion; spacing thresholds alone are insufficient."
        ),
    },
    {
        "candidate": "tracked_harmonic_order",
        "missing_rule": "admissibility rule over harmonic indices",
        "question": (
            "Does hardware or demodulation logic permit/exclude harmonic orders by modular or factor structure?"
        ),
        "needed_evidence": (
            "Tracking/demodulation documentation beyond the first-three-harmonics approximation."
        ),
    },
])

next_evidence


## 10. Current conclusion


In [ ]:
direct_count = int(hypothesis_audit["direct_gdt_ready"].sum())

if direct_count > 0:
    overall_status = "direct_application_established"
else:
    overall_status = "discrete_readout_structure_present_direct_GDT_not_established"

status = {
    "notebook_id": NOTEBOOK_ID,
    "status": overall_status,
    "engineering_domain": "TES microwave SQUID multiplexed readout",
    "source_id": "SOURCE_05",
    "candidate_mapping_count": len(candidate_mappings),
    "physically_justified_mapping_count": direct_count,
    "direct_application_claim_allowed": direct_count > 0,
    "strongest_current_candidates": [
        "analysis_subband_index",
        "resonator_grouping",
        "tracked_harmonic_order",
    ],
    "rejected_candidate_encodings": [
        "frequency_grid_by_khz_scaling",
        "flux_quanta_per_ramp",
    ],
    "current_conclusion": (
        "SOURCE_05 supplies genuine discrete readout structure and physical exclusion rules, "
        "but no candidate currently supplies both a physically justified residue-class rule "
        "and a coprimality/factor-exclusion rule."
    ),
    "next_research_step": (
        "Inspect firmware, resonator-frequency-plan, mask-layout, synchronization, or aliasing "
        "documentation for an explicit periodic index-allocation rule and exact factor-exclusion rule."
    ),
}

status


## 11. Becker-facing interpretation

The current defensible statement is:

> I tested the General Divisor Theorem against a genuinely discrete part of the TES readout architecture rather than against continuous fabrication variables. The SMuRF / µMUX system provides real integer-indexed structures and real channel exclusions, but the published SOURCE_05 evidence still does not give the theorem's required residue-class and coprimality rules. I therefore rejected arbitrary integer encodings and identified the firmware/frequency-plan rules that would be needed for a direct admissible-state calculation.

A later positive result would be stronger:

> Given a documented readout index rule \(n\equiv a\pmod m\) and compatibility rule \(\gcd(n,N)=1\), I used the GDT to compute the exact admissible channel/readout states, their period, count, and density.


## 12. Write outputs


In [ ]:
candidate_csv = OUTPUT_DIR / "candidate_gdt_mappings.csv"
hypothesis_csv = OUTPUT_DIR / "gdt_hypothesis_audit.csv"
results_csv = OUTPUT_DIR / "gdt_readout_results.csv"
exclusion_csv = OUTPUT_DIR / "physical_exclusion_audit.csv"
next_csv = OUTPUT_DIR / "next_evidence_requirements.csv"
status_json = OUTPUT_DIR / "gdt_readout_status.json"

candidate_mapping_df.to_csv(candidate_csv, index=False)
hypothesis_audit.to_csv(hypothesis_csv, index=False)
gdt_result_df.to_csv(results_csv, index=False)
physical_exclusion_audit.to_csv(exclusion_csv, index=False)
next_evidence.to_csv(next_csv, index=False)

status_json.write_text(
    json.dumps(
        status,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    ),
    encoding="utf-8",
)

written_files = {
    "candidate_gdt_mappings": candidate_csv,
    "gdt_hypothesis_audit": hypothesis_csv,
    "gdt_readout_results": results_csv,
    "physical_exclusion_audit": exclusion_csv,
    "next_evidence_requirements": next_csv,
    "gdt_readout_status": status_json,
}

for name, path in written_files.items():
    print(f"{name:28} {path.relative_to(ROOT)}")


## 13. Build export ZIP


In [ ]:
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for path in written_files.values():
    shutil.copy2(path, EXPORT_DIR / path.name)

if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()

with zipfile.ZipFile(
    EXPORT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(EXPORT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print("Export package:", EXPORT_ZIP)
print("Size:", f"{EXPORT_ZIP.stat().st_size:,}", "bytes")

try:
    from google.colab import files
    files.download(str(EXPORT_ZIP))
except ImportError:
    print("Automatic download is available only in Google Colab.")


## 14. Handoff

The next move depends on the result:

```text
SOURCE_05
   ↓
multiplexed_readout Engineering Object
   ↓
NB_06 candidate mappings
   ├── direct hypotheses found
   │       ↓
   │   exact GDT admissible-state calculation
   │
   └── hypotheses still missing
           ↓
       targeted firmware / layout / frequency-plan source search
```

At this stage, the most promising research target is no longer another generic TES paper. It is documentation that specifies **how discrete readout indices are actually assigned and excluded**.
